# Flowers Knowledge RAG Pipeline
Step-by-step RAG demo using LangChain + Groq + HuggingFace Embeddings + ChromaDB

## Step 1 - Install Dependencies

In [1]:
%pip install langchain langchain-community langchain-groq langchain-chroma langchain-text-splitters langchain-huggingface sentence-transformers groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Step 2 - Imports

In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

print('All libraries imported successfully')

All libraries imported successfully


## Step 3 - Load API Key

In [3]:
load_dotenv(override=True)
api_key = os.environ.get('GROQ_API_KEY')
print('Groq API Key loaded:', 'OK' if api_key else 'NOT FOUND - check your .env file')

Groq API Key loaded: OK


## Step 4 - Load flowers.txt (Knowledge Base)

In [4]:
loader = TextLoader('flowers.txt', encoding='utf-8')
documents = loader.load()

print(f'Loaded {len(documents)} document(s)')
print(f'Total characters: {len(documents[0].page_content)}')

Loaded 1 document(s)
Total characters: 36070


## Step 5 - Split Documents into Chunks

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(documents)

print(f'Total chunks created: {len(docs)}')
print(f'\nSample chunk:\n{docs[0].page_content[:300]}')

Total chunks created: 55

Sample chunk:
Flowers, also known as blooms and blossoms, are the reproductive structures of flowering plants. Typically, they are structured in four circular levels around the end of a stalk. These include: sepals, which are modified leaves that support the flower; petals, often designed to attract pollinators; 


## Step 6 - Create Vector Database with HuggingFace Embeddings

In [6]:
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

print('Vector database created successfully')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector database created successfully


## Step 7 - Initialize Groq LLM

In [7]:
llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.3,
    api_key=api_key
)

print('Groq LLM initialized')

Groq LLM initialized


## Step 8 - Build the RAG Function

In [8]:
def combine_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

def ask_rag(question):
    retrieved_docs = retriever.invoke(question)
    context = combine_docs(retrieved_docs)
    prompt = f"""
You are a factual question-answering assistant.
Use ONLY the information provided in the context.
If the answer is not present, say: I don't know.

Context:
{context}

Question: {question}

Answer:
"""
    response = llm.invoke(prompt)
    return response.content

print('RAG function ready')

RAG function ready


## Step 9 - Ask Questions

In [9]:
question = 'What is the purpose of a flower?'
answer = ask_rag(question)
print(f'Q: {question}')
print(f'A: {answer}')

Q: What is the purpose of a flower?
A: The main purpose of a flower is reproduction of the individual, aiding in the survival of the species.


In [10]:
question = 'What are the parts of a flower?'
answer = ask_rag(question)
print(f'Q: {question}')
print(f'A: {answer}')

Q: What are the parts of a flower?
A: The four main parts of a flower are: 
1. Sepals (modified leaves that support the flower)
2. Petals (often designed to attract pollinators)
3. Male parts (where pollen is presented)
4. Female parts (where pollen is received and its movement is facilitated to the egg).


In [11]:
question = 'What is pollination?'
answer = ask_rag(question)
print(f'Q: {question}')
print(f'A: {answer}')

Q: What is pollination?
A: Pollination is the movement of pollen from the male parts to the female parts, either between flowers of the same plant (self-pollination) or between flowers of different plants (cross-pollination).
